# 13. SQL Pivot & Unpivot: Matrix Transformations: Beginner Guide

### 📝 SQL Execution Order:
```text
┌─ SQL Query Execution Order (Sequential Pipeline) ────────────────────────────┐
│ 1. FROM & JOIN (Load)    ➔ 2. WHERE (Filter)       ➔ 3. GROUP BY (Bucket)    │
│ ➔ 4. HAVING (Agg Filter) ➔ 5. SELECT (Pick Cols)   ➔ 6. DISTINCT (Dedup)     │
│ ➔ 7. ORDER BY (Sort)     ➔ 8. LIMIT / OFFSET (Page)                          │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **13. SQL Pivot & Unpivot: Matrix Transformations**. Analytical reporting often requires reshaping normalized relational transaction logs into cross-tabular matrices (rows $\to$ columns) or unpivoting wide spreadsheets back into normalized relational streams (columns $\to$ rows). This notebook covers cross-database conditional aggregation pivoting (`SUM(CASE WHEN ...)`), unpivoting via `UNION ALL`, and dynamic matrix reshaping patterns.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Universal Conditional Aggregation Pivoting: `SUM(CASE WHEN col = 'Val' THEN ...)`
- [x] 🔹 Multi-Metric Matrix Summaries across Dimensions
- [x] 🔹 Normalizing Wide Datasets: Unpivoting via `UNION ALL`
- [x] 🔍 Scenario: Multi-Region Monthly Spend Breakdown Executive Heatmap








In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Conditional Aggregation Pivoting: `SUM(CASE ...)`
- **What it does:** Converts discrete categorical row values into dedicated columnar metrics using aggregate `CASE` expressions.
- **Syntax:** `SELECT grp, SUM(CASE WHEN cat = 'A' THEN val ELSE 0 END) AS val_a FROM table GROUP BY grp;`
- **Dataset Application & Code Demonstration:** Pivots transaction volume by card type across geographical regions.


In [2]:
%%sql
SELECT 
    region,
    ROUND(SUM(CASE WHEN card_type = 'Visa' THEN transaction_amount ELSE 0 END), 2) AS visa_volume,
    ROUND(SUM(CASE WHEN card_type = 'Mastercard' THEN transaction_amount ELSE 0 END), 2) AS mastercard_volume,
    ROUND(SUM(CASE WHEN card_type = 'Amex' THEN transaction_amount ELSE 0 END), 2) AS amex_volume,
    ROUND(SUM(CASE WHEN card_type = 'Discover' THEN transaction_amount ELSE 0 END), 2) AS discover_volume,
    ROUND(SUM(transaction_amount), 2) AS total_regional_volume
FROM transactions
WHERE region IS NOT NULL
GROUP BY region
ORDER BY total_regional_volume DESC;


,region,visa_volume,mastercard_volume,amex_volume,discover_volume,total_regional_volume
0,East,845794.57,0.0,874838.56,878257.05,3483441.29
1,West,822195.30,0.0,904135.92,872430.53,3458096.29
2,North,840960.59,0.0,804681.83,892395.55,3374657.98
3,South,867887.40,0.0,850038.89,814207.04,3362996.71
4,south,23328.55,0.0,25906.48,19715.11,95901.77
5,east,19095.18,0.0,18996.09,27694.92,93389.78
6,South,18333.59,0.0,33336.70,16763.04,93212.31
7,west,17927.98,0.0,18252.13,27337.86,81714.88
8,North,15543.75,0.0,13086.31,29239.47,77668.31
9,West,16981.20,0.0,18828.50,19231.14,71931.40


### 🔹 Unpivoting Wide Tables to Long Relational Format
- **What it does:** De-normalizes wide summary tables by transforming multiple columns into name-value attribute row pairs.
- **Syntax:** Combines columnar attributes using stacked `UNION ALL` branches.
- **Dataset Application & Code Demonstration:** Unpivots quarterly revenue projections.


In [3]:
%%sql
CREATE TABLE IF NOT EXISTS merchant_quarterly_targets (
    merchant_id TEXT PRIMARY KEY,
    q1_target REAL,
    q2_target REAL,
    q3_target REAL
);
INSERT OR REPLACE INTO merchant_quarterly_targets VALUES ('MERCH_501', 50000.0, 75000.0, 90000.0);

SELECT merchant_id, 'Q1' AS quarter, q1_target AS target_amount FROM merchant_quarterly_targets
UNION ALL
SELECT merchant_id, 'Q2' AS quarter, q2_target AS target_amount FROM merchant_quarterly_targets
UNION ALL
SELECT merchant_id, 'Q3' AS quarter, q3_target AS target_amount FROM merchant_quarterly_targets;


'Query Executed Successfully.'

## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Multi-Dimensional Pivot Matrix with Fraud Ratios
- **Objective:** Construct a matrix of total spend and fraud count per region across device types.
- **Approach:** Use conditional aggregation to generate multi-dimensional matrix metrics.


In [4]:
%%sql
SELECT 
    region,
    ROUND(SUM(CASE WHEN device_type = 'Mobile' THEN transaction_amount ELSE 0 END), 2) AS mobile_spend,
    SUM(CASE WHEN device_type = 'Mobile' THEN is_fraud ELSE 0 END) AS mobile_fraud_count,
    ROUND(SUM(CASE WHEN device_type = 'Desktop' THEN transaction_amount ELSE 0 END), 2) AS desktop_spend,
    SUM(CASE WHEN device_type = 'Desktop' THEN is_fraud ELSE 0 END) AS desktop_fraud_count
FROM transactions
WHERE region IS NOT NULL
GROUP BY region;


,region,mobile_spend,mobile_fraud_count,desktop_spend,desktop_fraud_count
0,East,20456.19,0,20978.01,2
1,North,16357.67,3,22178.02,1
2,South,26384.50,4,22409.11,5
3,West,21241.32,0,11994.96,0
4,East,900196.92,113,843945.56,90
5,North,829565.76,96,843106.38,87
6,South,855077.07,98,801520.89,85
7,West,830170.09,78,841815.89,84
8,east,19188.19,3,23948.91,5
9,north,11210.69,2,24688.30,4
